# Quickstart Notebook

This notebook demonstrates a simple end-to-end `trace_lib` run. Note that this version focuses on an offline toy-model run (works without downloads).


## Caveat on toy results

The toy run is a pipeline smoke test only. It validates that `Audit -> SAE training -> trace_features` works, but it is not a safety conclusion.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "trace_lib").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/trace_lib")


repo_root = find_repo_root(Path.cwd())
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import trace_lib as trace
from trace_lib.datasets import get_dataset, list_datasets

repo_root


In [ ]:
print("Available datasets:", list_datasets())
ds = get_dataset("harmful_refusal_v1")
print("harmful_refusal_v1 safety prompts:", len(ds.safety_prompts))
print("harmful_refusal_v1 baseline prompts:", len(ds.baseline_prompts))


## Build a tiny local model pair

The classes below are intentionally minimal and only exist to exercise the trace pipeline locally.


In [ ]:
import copy
import torch
import torch.nn as nn


class Batch(dict):
    def to(self, device):
        return Batch({k: v.to(device) for k, v in self.items()})


class SimpleTokenizer:
    def __init__(self, vocab_size=128):
        self.vocab_size = vocab_size
        self.pad_token = "<pad>"
        self.eos_token = "<eos>"

    def _encode(self, text, max_length):
        token_ids = [((ord(ch) % (self.vocab_size - 1)) + 1) for ch in text[:max_length]]
        return token_ids if token_ids else [1]

    def __call__(self, texts, return_tensors="pt", padding=True, truncation=True, max_length=64):
        encoded = [self._encode(t, max_length) for t in texts]
        seq_len = max(len(x) for x in encoded)
        if truncation:
            seq_len = min(seq_len, max_length)

        ids = []
        masks = []
        for seq in encoded:
            seq = seq[:seq_len]
            mask = [1] * len(seq)
            if padding and len(seq) < seq_len:
                pad_len = seq_len - len(seq)
                seq = seq + [0] * pad_len
                mask = mask + [0] * pad_len
            ids.append(seq)
            masks.append(mask)

        return Batch({
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(masks, dtype=torch.long),
        })


class ToyBackbone(nn.Module):
    def __init__(self, hidden_size=32, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(num_layers)])


class ToyModel(nn.Module):
    def __init__(self, vocab_size=128, hidden_size=32, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.model = ToyBackbone(hidden_size=hidden_size, num_layers=num_layers)
        self.config = type("Config", (), {"hidden_size": hidden_size})()

    def forward(self, input_ids, attention_mask=None):
        x = self.embed(input_ids)
        for layer in self.model.layers:
            x = torch.tanh(layer(x))
        return x


base_model = ToyModel(hidden_size=32, num_layers=2)
quant_model = copy.deepcopy(base_model)

with torch.no_grad():
    for param in quant_model.parameters():
        param.copy_(torch.round(param * 8) / 8)

tokenizer = SimpleTokenizer()


## Run a simple trace audit


In [ ]:
audit = trace.Audit(
    base_model=base_model,
    quant_model=quant_model,
    tokenizer=tokenizer,
    device="cpu",
)

dataset = get_dataset("harmful_refusal_v1")
training_prompts = dataset.safety_prompts[:20] + dataset.baseline_prompts[:20]

_ = audit.train_sae(
    layer=0,
    prompts=training_prompts,
    d_sae=64,
    num_steps=25,
    batch_size=8,
)

report = audit.trace_features(dataset="harmful_refusal_v1", layers=[0])

print(f"CPFR score: {report.cpfr_score:.4f}")
print(f"Integrity status: {report.integrity_status}")
print(f"Degraded features: {len(report.degraded_features)}")
print(f"Lost features: {len(report.lost_features)}")


In [ ]:
print(report.summary())
output_path = repo_root / "examples" / "audit_report_toy.json"
report.save(output_path)
print("Saved report to:", output_path)


## Expected outputs

For the toy path, expect output like:
- `CPFR score`: usually around `0.60` to `0.75`
- `Integrity status`: often `DEGRADED`
- non-zero `Degraded features` and `Lost features`

You should also see:
- a rich text summary table from `report.summary()`
- a saved JSON file at `examples/audit_report_toy.json`

This is expected because the toy "quantized" model is created by coarse weight rounding, so it intentionally introduces feature drift. The exact numbers can vary run-to-run.
